RDM Logic:

In [1]:
import numpy as np
from scipy import stats

def build_rdm(rating_matrix):
    """
    Takes a (1000, 34) rating matrix and returns a 34x34 RDM, where dissimilarity is defined as {1 - correlation} between each pair of attribute columns.
    """

    rm_transposed = np.transpose(rating_matrix)
    correlations = np.corrcoef(rm_transposed)
    dissimilarities = np.subtract(1, correlations)
    
    return dissimilarities

def get_unique_correlations(rdm):
    """
    Extracts the upper triangle (excluding the diagonal) from an RDM and returns it as a flat vector, so it can later be used to get the final human-vs-model RSA score.
    """

    rows, cols = rdm.shape

    indices = np.triu_indices(n=rows, k=1, m=cols) # k=1 excludes the diagonal
    flattened = rdm[indices] # extracts the upper triangle as a flat vector

    return flattened

def compare_rdms(rdm_1, rdm_2):
    """
    Returns the correlation coefficient for the two RDMS.
    """

    rdm_1_vector = get_unique_correlations(rdm_1)
    rdm_2_vector = get_unique_correlations(rdm_2)

    spearman_corr, p_val = stats.spearmanr(rdm_1_vector, rdm_2_vector)
    
    return spearman_corr, p_val

Data Analysis:

In [2]:
import pandas as pd

gemini_df = pd.read_csv("/work/outputs/google_gemini-2.5-flash-lite/neutral.csv")
gemini_rdm = build_rdm(gemini_df.drop(columns=['stimulus']).to_numpy())

claude_df = pd.read_csv("/work/outputs/anthropic_claude-sonnet-5/neutral.csv")
claude_rdm = build_rdm(claude_df.drop(columns=['stimulus']).to_numpy())

gpt_df = pd.read_csv("/work/outputs/openai_gpt-5.4-mini/neutral.csv")
gpt_rdm = build_rdm(gpt_df.drop(columns=['stimulus']).to_numpy())

human_df = pd.read_csv("/work/20260629-213612/omi-main/attribute_means.csv")
human_rdm = build_rdm(human_df.drop(columns=['stimulus']).to_numpy())

gemini_corr, gemini_pval = compare_rdms(human_rdm, gemini_rdm)
claude_corr, claude_pval = compare_rdms(human_rdm, claude_rdm)
gpt_corr, gpt_pval = compare_rdms(human_rdm, gpt_rdm)

print(f"Gemini Corr: {gemini_corr}; Gemini Pval: {gemini_pval}")
print(f"GPT Corr: {gpt_corr}; GPT Pval: {gpt_pval}")
print(f"Claude Corr: {claude_corr}; Claude Pval: {claude_pval}")

Plotting:

In [7]:
import matplotlib.pyplot as plt

def plot_dissimilarities(rdm_1, rdm_2, name_1, name_2):
    """
    Plots human and ai dissimilarities for each attribute pair.
    If share close similarity, should appear closer to a tight, increasing line {y=x}
    If tighter on left side, more similar on strongly-related pairs
    If tighter on right side, more similar on high dissimilarity pairs
    """
    dissimilarities_1 = get_unique_correlations(rdm_1)
    dissimilarities_2 = get_unique_correlations(rdm_2)
    
    plt.scatter(dissimilarities_1, dissimilarities_2, alpha=0.5)
    plt.xlabel(f"{name_1} RDM dissimilarity")
    plt.ylabel(f"{name_2} RDM dissimilarity")
    plt.title(f"{name_1} vs {name_2}: attribute-pair dissimilarities")
    plt.show()


plot_dissimilarities(human_rdm, gemini_rdm, "Human", "Gemini")
plot_dissimilarities(human_rdm, gpt_rdm, "Human", "GPT")
plot_dissimilarities(human_rdm, claude_rdm, "Human", "Claude")

In [5]:
#Simple barchart comparing correlation values

models = ['Gemini 2.5 Flash-Lite', 'GPT-5.4 Mini', 'Claude Sonnet 5']
correlations = [gemini_corr, gpt_corr, claude_corr]

plt.figure(figsize=(6,4))
plt.bar(models, correlations, color=['#4285F4', '#10A37F', '#D97757'])
plt.ylabel("Spearman ρ (vs. human RDM)")
plt.title("Model-Human RSA Alignment by Model")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("rsa_correlations.png", dpi=150)
plt.show()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=db20d035-0f07-44b3-9147-5cdb9d6d1fdd' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>